# Transformer + PPO: fixed-deck self-play

This is the new neural trainer. Notebook 11 and all its saved linear models remain separate.

The model learns card/board interactions with a Transformer, scores **only legal moves**, and estimates the game outcome. PPO trains one learner seat per game against either a frozen copy of the current policy (25%) or a uniformly sampled archived checkpoint (75%). The initial random-weight neural policy remains in the archive. No pretrained strong opponents are bundled.

Four fixed, published lists from [vS report 358](https://www.vicioussyndicate.com/vs-data-reaper-report-358/) match the simulator's frozen September 18, 2026 catalog: Cannoneer Dragon Warrior, Bite Attack Druid, Thalena Egg Death Knight and Stealth Herald Rogue. Their strengths refer to that historical patch, not the current live game. No random decks or substitutions.

**The setup and preflight cells do not train.** The launch cell stays disabled until you set `START_TRAINING = True`. Do not use Run All to start unintentionally. Each completed PPO iteration saves a resumable checkpoint; capped games are counted and discarded without a weight update from them. If all games cap, the iteration saves unchanged weights. Simulator approximations still apply. Strong real-game play has not been demonstrated.

If PyTorch is missing, run this yourself in a separate cell, then restart this kernel:
```python
%pip install -e ".[neural,notebook]"
```


In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'tools/train_neural.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the death-knight-lab checkout.')
START_TRAINING = False  # Change to True ONLY when you want to begin learning.
ITERATIONS = 1
GAMES_PER_ITERATION = 8
MAX_ACTIONS = 1000
SEED = 71
DEVICE = 'cpu'  # cpu is the portable default; cuda / mps / auto are optional.
THREADS = 2
RESUME_CHECKPOINT = None  # Absolute path to training-000001.pt from THIS neural trainer.
OPPONENT_CHECKPOINTS = []  # Optional compatible neural .pt files; leave empty when resuming.
base_command = [sys.executable, str(ROOT / 'tools/train_neural.py'),
    '--iterations', str(ITERATIONS), '--games-per-iteration', str(GAMES_PER_ITERATION),
    '--max-actions', str(MAX_ACTIONS), '--seed', str(SEED), '--device', DEVICE,
    '--threads', str(THREADS)]
print('Setup only. Training has not started. Interpreter:', sys.executable)



In [ ]:
# Safe read-only preflight: validates decks, dependencies and architecture; starts no games.
check = subprocess.run(base_command + ['--preflight'], cwd=ROOT, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(check.stdout)
if check.returncode:
    raise RuntimeError('Preflight failed. Fix the message above before launching.')


## Start only when you choose

Set `START_TRAINING = True` in the setup cell, rerun that cell, then run the cell below. A fresh run starts from random neural weights. Your old linear-policy JSON cannot be loaded into this different architecture.

To resume, set `RESUME_CHECKPOINT` to the printed `.pt` path, keep the seed, deck panel, device, thread count, games per iteration, model and PPO settings unchanged. `ITERATIONS` means **additional** iterations. A new run folder preserves the old one and copies the archived opponents. Checkpoint files require their companion `.json` hash receipts; training resume also requires the sibling opponent files and receipts. Keep the entire run folder.

Interrupt the notebook to stop the child process. The last fully saved iteration remains resumable; work in the interrupted iteration is lost. Logs report capped games, PPO loss, entropy, KL and actual optimizer steps. These are diagnostics, not evidence of playing strength. Increasing self-play win rate alone does not measure progress against an independent opponent.


In [ ]:
if not START_TRAINING:
    print('Training is disabled. Set START_TRAINING = True and rerun the setup cell when ready.')
else:
    command = base_command + ['--train']
    if RESUME_CHECKPOINT is not None:
        command += ['--resume', str(RESUME_CHECKPOINT)]
    for checkpoint in OPPONENT_CHECKPOINTS:
        command += ['--opponent', str(checkpoint)]
    LAST_CHECKPOINT = None
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                try:
                    event = json.loads(line)
                except ValueError:
                    print(line.rstrip())
                    continue
                kind = event.get('event')
                if kind == 'started':
                    print('Run folder:', event['folder'])
                    print('Device:', event['device'], '| Parameters:', event['parameters'])
                elif kind == 'game_started':
                    print(f"Iteration {event['iteration']}: game {event['game']}/{event['total']} "
                          f"| {event['learner_deck']} vs {event['opponent_deck']} | {event['opponent']}")
                elif kind == 'game_finished':
                    print(f"  {event['actions']} actions | capped={event['truncated']} | reward={event['reward']}")
                elif kind == 'iteration_saved':
                    LAST_CHECKPOINT = event['checkpoint']['path']
                    print('PPO:', event['metrics'])
                    print('Resume checkpoint:', LAST_CHECKPOINT)
                elif kind == 'complete':
                    print('Run completed. Final checkpoint:', event['checkpoint']['path'])
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise RuntimeError('Run stopped early. Resume from the last saved checkpoint; do not delete the run folder.')


## What this does and does not establish

The encoder consumes policy-visible fields, visible printed card definitions, all legal candidates and the latest 64 public events (with an omitted-history count). It does not access the opponent's private hand, deck order, simulator RNG or replay headers. Field hashing can collide and recent history is bounded; it is not a perfect belief-state model. Token/field overflow fails explicitly rather than dropping legal actions or board entities.

Training uses terminal win/draw/loss rewards, per-learner-decision GAE, clipped policy/value objectives, entropy regularization, gradient clipping and KL early stopping. No reward is invented for an action-capped game. A draw can still train the value estimate. Only learner-seat decisions enter PPO; archived opponents remain frozen throughout collection. Match scheduling balances deck pairs, learner seats and first-player seats over complete cycles.

This is a runnable research baseline, not AlphaStar or a certified strong player. Future strength evaluation should use fixed held-out seeds and independent opponents; hidden-information search, richer belief modeling, human replay pretraining and further simulator fidelity work remain separate improvements.
